In [1]:
using LinearAlgebra
using BlackBoxOptim
using HDF5

Fidelity for qubit can be written:

$F(\rho, \sigma) = \operatorname{tr}(\rho \sigma) + 2\sqrt{\det(\rho) \det(\sigma)}$

In [2]:
function infidelity_norm(ρ, σ)
    (real(1 - tr(ρ * σ) - 2*sqrt(det(ρ)*det(σ))))^2
end

infidelity_norm (generic function with 1 method)

In [5]:
function read_timeevolution(file_name, state, γ)
    h5open(file_name, "r") do file
        ρᵧ = read(file[state][string(γ)])
        t = ρᵧ["t"]
        ρ₀₀ = ρᵧ["p0"]; Re_ρ₀₁ = ρᵧ["s_re"];  Im_ρ₀₁ = ρᵧ["s_im"]
        ρ_series = []
        t_series = []

        for i in 1:length(t)
            ρᵢ= [ ρ₀₀[i]                      Re_ρ₀₁[i] + im * Im_ρ₀₁[i]
                  Re_ρ₀₁[i] - im * Im_ρ₀₁[i]  1 - ρ₀₀[i]                 ]
            push!(ρ_series, convert(Matrix{ComplexF64}, ρᵢ))
            push!(t_series, convert(Float64, t[i]))
        end
        return(t_series, ρ_series)
    end
end

# Define the Pauli matrices
σˣ = [0 1; 1 0]
σʸ = [0 im; -im 0]
σᶻ = [1 0; 0 -1]

# Define the basis elements
fᴷ₁ = σˣ / 2
fᴷ₂ = σʸ / 2
fᴷ₃ = σᶻ / 2

# Check orthogonality and normalization
@assert tr(fᴷ₁ * fᴷ₂) ≈ 0
@assert tr(fᴷ₁ * fᴷ₃) ≈ 0
@assert tr(fᴷ₂ * fᴷ₃) ≈ 0
@assert tr(fᴷ₁ * fᴷ₁) ≈ 1 / 2
@assert tr(fᴷ₂ * fᴷ₂) ≈ 1 / 2
@assert tr(fᴷ₃ * fᴷ₃) ≈ 1 / 2

fᴼᴺᴮ = [fᴷ₁, fᴷ₂, fᴷ₃]

# Function to calculate Dc
function Dc(ρ, t, H, C)
    U = (H * ρ - ρ * H) / im
    D = sum(C .* [2 * fᵢ * ρ * fⱼ' - ρ * fⱼ' * fᵢ - fⱼ' * fᵢ * ρ for fᵢ in fᴼᴺᴮ, fⱼ in fᴼᴺᴮ]) / 2
    return U + D
end

function construct_H(params)
    ϵ, h_Re, h_Im = params[1:3]
    return [
        ϵ               h_Re + im * h_Im
        h_Re - im * h_Im -ϵ
    ] / 2
end

function construct_C(params)
    m_Re = reshape(params[4:12], (3, 3))
    m_Im = reshape(params[13:21], (3, 3))
    M = m_Re + im * m_Im
    return M * M'
end


# Define the objective function for optimization
function kossak_obj(params, ρ, t)
    H = construct_H(params)
    C = construct_C(params)

    obj = 0.0
    for i in 3:length(ρ)
        ρ1 = ρ[i]
        ρ2 = ρ[i - 2] + (t[i] - t[i - 1]) * (Dc(ρ[i], t[i], H, C) + 4 * Dc(ρ[i - 1], t[i - 1], H, C) + Dc(ρ[i - 2], t[i - 2], H, C)) / 3
        obj += infidelity_norm(ρ1,ρ2)
    end
    return obj
end

kossak_obj (generic function with 1 method)

In [6]:
# Define initial parameters
initial_params = [0.0, 0.0, 0.0,  # H parameters: ϵ, h_Re, h_Im
                  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,  # m_Re parameters
                  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]; # m_Im parameters

initial_params .+= 1;

In [7]:
# Define the objective function wrapper
function objective(params)

    objGEXY = kossak_obj(params, ρᵍ, tᵍ) + kossak_obj(params, ρᵉ, tᵉ) + kossak_obj(params, ρˣ, tˣ) + kossak_obj(params, ρʸ, tʸ)

    return objGEXY
end

objective (generic function with 1 method)

In [8]:
# Define the density matrix evolution and time points

file_name = "../DATA/ALL_GAMMAS_B4_D10.h5"

#γ = [ "0.079477",  "0.25133", "0.79477", "2.5133", "7.9477", "25.133", "79.477", "251.33"]

γ = "0.25133"
γᶠ = parse(ComplexF64, γ)

tᵍ, ρᵍ = read_timeevolution(file_name, "B1", γ)
tᵉ, ρᵉ = read_timeevolution(file_name, "B2", γ)
tˣ, ρˣ = read_timeevolution(file_name, "B3", γ)
tʸ, ρʸ = read_timeevolution(file_name, "B4", γ);

In [10]:
# Custom stopping condition
function stop_condition(optstate)
    best_fitness = best_fitness(optstate)
    if best_fitness <= 2e-8
        println("Stopping condition met: fitness <= 2e-8")
        return true
    end
    return false
end

stop_condition (generic function with 1 method)

In [11]:
# Setup the BlackBoxOptim optimizer
res = bboptimize(objective, 
SearchRange = (-25.0, 25.0), 
NumDimensions = length(initial_params), 
Method = :generating_set_search, #:adaptive_de_rand_1_bin_radiuslimited
MaxSteps = 1000,
Callback = stop_condition,
autodiff = :forward,
TraceMode = :verbose)

Starting optimization with optimizer GeneratingSetSearcher(BlackBoxOptim.ConstantDirectionGen)
0.00 secs, 1 evals, 0 steps, fitness=920861.124446394
1.46 secs, 9 evals, 1 steps, fitness=871554.042155966
2.18 secs, 13 evals, 4 steps, fitness=585579.820378627
2.70 secs, 16 evals, 6 steps, fitness=480678.295555905
3.56 secs, 21 evals, 7 steps, fitness=480218.094663428
5.19 secs, 30 evals, 9 steps, fitness=274032.847597023
5.71 secs, 33 evals, 12 steps, fitness=95576.045576265
6.42 secs, 37 evals, 13 steps, fitness=95046.181171056
7.67 secs, 44 evals, 14 steps, fitness=44624.795591037
9.23 secs, 53 evals, 15 steps, fitness=41171.373218671
11.37 secs, 65 evals, 16 steps, fitness=30675.675990642
12.25 secs, 70 evals, 18 steps, fitness=21829.055719847
15.56 secs, 88 evals, 21 steps, fitness=20879.858833577
16.16 secs, 91 evals, 22 steps, fitness=19773.615865565
17.22 secs, 96 evals, 24 steps, fitness=16749.234486245
19.61 secs, 109 evals, 25 steps, fitness=16720.266558896
20.53 secs, 114 eval

BlackBoxOptim.OptimizationResults("generating_set_search", "Max number of steps (1000) reached", 1001, 1.722026004621982e9, 5837.425911903381, ParamsDictChain[ParamsDictChain[Dict{Symbol, Any}(:RngSeed => 292068, :NumDimensions => 21, :autodiff => :forward, :SearchRange => (-25.0, 25.0), :TraceMode => :verbose, :Method => :generating_set_search, :Callback => stop_condition, :MaxSteps => 1000),Dict{Symbol, Any}()],Dict{Symbol, Any}(:CallbackInterval => -1.0, :TargetFitness => nothing, :TraceMode => :compact, :FitnessScheme => ScalarFitnessScheme{true}(), :MinDeltaFitnessTolerance => 1.0e-50, :NumDimensions => :NotSpecified, :FitnessTolerance => 1.0e-8, :TraceInterval => 0.5, :MaxStepsWithoutProgress => 10000, :MaxSteps => 10000…)], 26259, ScalarFitnessScheme{true}(), BlackBoxOptim.TopListArchiveOutput{Float64, Vector{Float64}}(1.093561559205762e-8, [25.0, -0.00357720411153295, 0.001174477179191058, 0.3286199580053726, 0.010900444228440165, -0.011671631019105178, 0.23058111499702605, 0.1

In [12]:
# Extract optimized parameters
optimized_params = best_candidate(res)

# Substitute the optimized parameters back into H and C
optimized_H = construct_H(optimized_params)
optimized_C = construct_C(optimized_params)

println("Optimized H: ", optimized_H)
println("Optimized C: ", optimized_C)

Optimized H: ComplexF64[12.5 + 0.0im -0.001788602055766475 + 0.000587238589595529im; -0.001788602055766475 - 0.000587238589595529im -12.5 + 0.0im]
Optimized C: ComplexF64[0.28423304602435595 + 0.0im -0.007569418013892519 + 0.21557279419362751im -0.00941430674358262 - 0.008636032863105433im; -0.007569418013892519 - 0.21557279419362751im 0.16369991574284648 + 0.0im -0.006299172235746539 + 0.007370142853267237im; -0.00941430674358262 + 0.008636032863105433im -0.006299172235746539 - 0.007370142853267237im 0.000574212736510471 + 0.0im]


In [13]:
optimized_H

2×2 Matrix{ComplexF64}:
       12.5+0.0im          -0.0017886+0.000587239im
 -0.0017886-0.000587239im       -12.5+0.0im

In [16]:
optimized_C

3×3 Matrix{ComplexF64}:
   2718.03+0.0im      -0.991235+2718.43im   11.7462+15.8111im
 -0.991235-2718.43im    2718.85+0.0im       15.8074-11.7529im
   11.7462-15.8111im    15.8074+11.7529im  0.149374+0.0im

In [10]:
search_range = (-25.0, 25.0)

# List of methods to compare
methods = [
    :adaptive_de_rand_1_bin_radiuslimited,
    :adaptive_de_rand_1_bin,
    :de_rand_1_bin_radiuslimited,
    :de_rand_1_bin,
    :de_best_1_bin,
    :de_rand_2_bin,
    :de_best_2_bin,
    :pbasis_de_rand_2_bin_radiuslimited,
    :pbasis_de_rand_2_bin,
    :adaptive_de_rand_2_bin_radiuslimited,
    :adaptive_de_rand_2_bin,
    :sep_de_rand_1_bin,
    :sep_de_rand_2_bin,
    :sep_de_best_1_bin,
    :sep_de_best_2_bin,
    :multi_strategy_adaptive_de,
    :multi_de_rand_1_bin,
    :multi_de_best_1_bin,
    :multi_de_rand_2_bin,
    :multi_de_best_2_bin,
    :best_1_bin_radiuslimited,
    :rand_1_bin_radiuslimited,
    :rand_2_bin_radiuslimited,
    :best_2_bin_radiuslimited,
    :best_1_bin,
    :rand_1_bin,
    :rand_2_bin,
    :best_2_bin,
    :self_adaptive_de_rand_1_bin,
    :self_adaptive_de_rand_2_bin,
    :self_adaptive_de_best_1_bin,
    :self_adaptive_de_best_2_bin,
    :gradient_de_rand_1_bin,
    :gradient_de_rand_2_bin,
    :gradient_de_best_1_bin,
    :gradient_de_best_2_bin
]

36-element Vector{Symbol}:
 :adaptive_de_rand_1_bin_radiuslimited
 :adaptive_de_rand_1_bin
 :de_rand_1_bin_radiuslimited
 :de_rand_1_bin
 :de_best_1_bin
 :de_rand_2_bin
 :de_best_2_bin
 :pbasis_de_rand_2_bin_radiuslimited
 :pbasis_de_rand_2_bin
 :adaptive_de_rand_2_bin_radiuslimited
 ⋮
 :best_2_bin
 :self_adaptive_de_rand_1_bin
 :self_adaptive_de_rand_2_bin
 :self_adaptive_de_best_1_bin
 :self_adaptive_de_best_2_bin
 :gradient_de_rand_1_bin
 :gradient_de_rand_2_bin
 :gradient_de_best_1_bin
 :gradient_de_best_2_bin

In [13]:
res_compare = compare_optimizers(objective; SearchRange = (-26, 26),
 NumDimensions =length(initial_params),
 methods=methods,
 max_steps=500,
 autodiff = :forward
 #MaxTime = 25.0
 );

In [ ]:
# Extract optimized parameters
optimized_params = best_candidate(res_compare)

# Substitute the optimized parameters back into H and C
optimized_H = construct_H(optimized_params)
optimized_C = construct_C(optimized_params)

println("Optimized H: ", optimized_H)
println("Optimized C: ", optimized_C)

In [ ]:
optimized_H

In [ ]:
optimized_C

In [ ]:
# Setup the BlackBoxOptim optimizer
res = bboptimize(objective, SearchRange = (-25.0, 25.0), NumDimensions = length(initial_params), Method = :adaptive_de_rand_1_bin_radiuslimited, MaxSteps = 10000)

# Extract optimized parameters
optimized_params = best_candidate(res)

# Substitute the optimized parameters back into H and C
optimized_H = construct_H(optimized_params)
optimized_C = construct_C(optimized_params)

println("Optimized H: ", optimized_H)
println("Optimized C: ", optimized_C)

In [8]:
optimized_H

2×2 Matrix{ComplexF64}:
 12.2714+0.0im       1.34831+9.72057im
 1.34831-9.72057im  -12.2714+0.0im

In [9]:
optimized_C

3×3 Matrix{ComplexF64}:
  2454.08+0.0im       38.6602+2449.37im  -76.4486-15.0218im
  38.6602-2449.37im   2446.43+0.0im      -19.7415+78.9891im
 -76.4486+15.0218im  -19.7415-78.9891im   27.3426+0.0im

In [ ]:
result_bbo = bboptimize(polyGEXY; SearchRange = (-26.0, 26.0), NumDimensions = 21,
    MaxTime = 20, 
    Method = :adaptive_de_rand_1_bin_radiuslimited,
    InitialPopulation = [initial_guess])

optimal_value_bbo = best_candidate(result_bbo)
minimum_function_value_bbo = best_fitness(result_bbo)